# ML System Design: Search Ranking

Applying the universal design template to build a search ranking system.

This notebook covers:
1. System design walkthrough for search
2. Query understanding (intent, expansion, correction)
3. Candidate retrieval (inverted index, BM25, ANN)
4. Learning-to-rank (pointwise, pairwise, listwise)
5. Implementation: BM25 from scratch
6. Implementation: Pairwise ranking loss (RankNet)

---

## Stage 1 — Problem Clarification

**Prompt**: "Design a search ranking system for an e-commerce platform."

### Clarifying Questions & Assumed Answers

| Question | Answer |
|----------|--------|
| What are we searching? | Product catalog (100M products) |
| Who is searching? | Consumers on web and mobile |
| Scale? | 50M DAU, ~10 searches/user/day → ~500M queries/day → ~6K QPS avg, ~20K peak |
| Latency? | < 200ms end-to-end (type-to-results) |
| Current solution? | Keyword matching with hand-tuned boosting |
| Goal? | Improve purchase rate from search, reduce zero-result rate |
| Query types? | Product names, categories, brands, natural language ("red dress for wedding") |

---
## Stage 2 — Metrics

### Offline Metrics

| Metric | What it Measures | Notes |
|--------|-----------------|-------|
| **NDCG@K** | Ranking quality with graded relevance | Primary offline metric (K=10) |
| **MRR** (Mean Reciprocal Rank) | Position of first relevant result | Good for navigational queries |
| **Recall@K** | Did we retrieve the relevant document? | For retrieval stage evaluation |
| **Precision@K** | What fraction of top-K are relevant? | Good for precision-critical queries |

### Online Metrics

| Metric | What it Measures |
|--------|------------------|
| **Click-through rate (CTR)** | % of searches where user clicks a result |
| **Conversion rate** | % of searches leading to purchase |
| **Revenue per search** | Direct business impact |
| **Zero-result rate** | % of queries with no results (should minimize) |
| **Dwell time** | Time spent on clicked result (proxy for satisfaction) |
| **Abandonment rate** | % of searches where user leaves without clicking |
| **Reformulation rate** | % of searches followed by another search (indicates dissatisfaction) |

### Guardrail Metrics
- Latency (p50, p99)
- Diversity of results (brand, price range, seller diversity)
- Fairness across sellers/brands

---
## Query Understanding

Before retrieving results, we need to understand the query.

### Intent Classification

| Intent | Example | Action |
|--------|---------|--------|
| **Navigational** | "Nike Air Max 90" | Direct product match, exact match boost |
| **Informational** | "best running shoes" | Broader retrieval, review/guide content |
| **Transactional** | "buy iPhone 15 pro" | Product results, price-focused |
| **Category** | "laptops" | Browse experience, faceted results |

Model: Fine-tuned text classifier (BERT or distilled model) on labeled query-intent data.

### Query Expansion

Enrich the query to improve recall:

1. **Synonym expansion**: "sofa" → "sofa OR couch OR settee" (from synonym dictionary or learned embeddings)
2. **Acronym expansion**: "TV" → "television"
3. **Related terms**: "iPhone" → also search for "phone case" (query suggestion, not expansion)
4. **Category mapping**: "red dress" → category:dresses, color:red

### Spell Correction

1. **Edit distance** (Levenshtein): find closest word in dictionary
2. **Noisy channel model**: P(correction | misspelling) ∝ P(misspelling | correction) * P(correction)
3. **Neural**: seq2seq model trained on (misspelling, correction) pairs from query logs
4. **Practical**: use query logs to build correction pairs (query that was reformulated → original was likely misspelled)

---
## Candidate Retrieval

### Inverted Index

The foundation of text search:

```
Term → [(doc_id, term_freq, positions), ...]

"running" → [(doc_42, 3, [5,12,45]), (doc_99, 1, [8]), ...]
"shoes"   → [(doc_42, 2, [6,46]), (doc_55, 5, [1,3,7,9,12]), ...]
```

Query "running shoes":
1. Look up posting lists for "running" and "shoes"
2. Intersect (AND) or union (OR) the posting lists
3. Score each document using a relevance function (TF-IDF, BM25)

### BM25 (Best Match 25)

The industry-standard relevance scoring function. An improvement over TF-IDF.

$$\text{BM25}(q, d) = \sum_{t \in q} \text{IDF}(t) \cdot \frac{f(t, d) \cdot (k_1 + 1)}{f(t, d) + k_1 \cdot (1 - b + b \cdot \frac{|d|}{\text{avgdl}})}$$

where:
- $f(t, d)$ = frequency of term $t$ in document $d$
- $|d|$ = document length
- $\text{avgdl}$ = average document length in corpus
- $k_1$ = term frequency saturation (typically 1.2-2.0)
- $b$ = length normalization (typically 0.75)
- $\text{IDF}(t) = \log \frac{N - n(t) + 0.5}{n(t) + 0.5}$

**Intuition**:
- IDF: rare terms are more important
- TF saturation: the 10th occurrence of a term adds less than the 1st (controlled by $k_1$)
- Length normalization: longer documents naturally have more term occurrences (controlled by $b$)

### Approximate Nearest Neighbors (ANN)

For semantic retrieval (not just keyword matching):

1. Encode query and documents into dense embeddings (using BERT, sentence-transformers, etc.)
2. At index time: build ANN index over document embeddings
3. At query time: encode query → ANN lookup → top-K nearest documents

Libraries: FAISS, ScaNN, Annoy, HNSW (hnswlib)

**Hybrid retrieval** (best practice):
```
Candidates = BM25_top_500 ∪ ANN_top_500 → deduplicate → ~700 candidates → Ranker
```

---
## Implementation: BM25 from Scratch

In [ ]:
import numpy as np
import math
from collections import Counter, defaultdict
from typing import List, Dict, Tuple


class BM25:
    """
    BM25 (Okapi BM25) implementation from scratch.
    
    BM25(q, d) = sum over terms t in q:
        IDF(t) * (f(t,d) * (k1 + 1)) / (f(t,d) + k1 * (1 - b + b * |d| / avgdl))
    
    where:
        f(t,d) = term frequency of t in document d
        |d| = length of document d (in tokens)
        avgdl = average document length
        k1 = term frequency saturation parameter (default 1.5)
        b = length normalization parameter (default 0.75)
        IDF(t) = log((N - n(t) + 0.5) / (n(t) + 0.5) + 1)
            N = total documents, n(t) = docs containing term t
    """
    
    def __init__(self, k1: float = 1.5, b: float = 0.75):
        self.k1 = k1
        self.b = b
        self.corpus_size = 0
        self.avgdl = 0.0
        self.doc_freqs = {}     # term -> number of docs containing term
        self.idf = {}           # term -> IDF score
        self.doc_len = []       # length of each document
        self.tf = []            # list of Counter (term freq for each doc)
    
    def _tokenize(self, text: str) -> List[str]:
        """Simple whitespace + lowercase tokenization."""
        return text.lower().split()
    
    def fit(self, corpus: List[str]) -> 'BM25':
        """Index a corpus of documents."""
        self.corpus_size = len(corpus)
        self.tf = []
        self.doc_len = []
        self.doc_freqs = defaultdict(int)
        
        total_len = 0
        for doc in corpus:
            tokens = self._tokenize(doc)
            self.doc_len.append(len(tokens))
            total_len += len(tokens)
            
            # Term frequency for this document
            tf_counter = Counter(tokens)
            self.tf.append(tf_counter)
            
            # Document frequency: count each term once per document
            for term in set(tokens):
                self.doc_freqs[term] += 1
        
        self.avgdl = total_len / self.corpus_size
        
        # Precompute IDF for all terms
        self.idf = {}
        for term, df in self.doc_freqs.items():
            # Robertson-Sparck Jones IDF with +1 for numerical stability
            self.idf[term] = math.log(
                (self.corpus_size - df + 0.5) / (df + 0.5) + 1.0
            )
        
        return self
    
    def score(self, query: str, doc_idx: int) -> float:
        """Score a single query-document pair."""
        query_terms = self._tokenize(query)
        doc_tf = self.tf[doc_idx]
        dl = self.doc_len[doc_idx]
        
        score = 0.0
        for term in query_terms:
            if term not in self.idf:
                continue  # term not in corpus
            
            tf = doc_tf.get(term, 0)
            idf = self.idf[term]
            
            # BM25 term score
            numerator = tf * (self.k1 + 1)
            denominator = tf + self.k1 * (1 - self.b + self.b * dl / self.avgdl)
            score += idf * numerator / denominator
        
        return score
    
    def search(self, query: str, top_k: int = 10) -> List[Tuple[int, float]]:
        """Return top-k (doc_index, score) pairs for a query."""
        scores = [(i, self.score(query, i)) for i in range(self.corpus_size)]
        scores.sort(key=lambda x: x[1], reverse=True)
        return scores[:top_k]

In [ ]:
# Test BM25 on a small product corpus
corpus = [
    "Nike Air Max 90 running shoes men black white comfortable lightweight",
    "Adidas Ultraboost 22 running shoes women pink breathable responsive cushioning",
    "New Balance 990v5 running shoes men grey classic heritage made in USA",
    "Nike Air Force 1 casual shoes men white leather classic sneaker",
    "Puma RS-X casual shoes women colorful retro style chunky sole",
    "Brooks Ghost 15 running shoes neutral cushioned road running daily trainer",
    "Nike Dunk Low casual shoes skateboarding leather suede retro style",
    "ASICS Gel-Kayano 29 stability running shoes overpronation support men",
    "Red formal dress women cocktail party evening elegant midi length",
    "Blue casual dress women summer cotton comfortable breathable lightweight",
    "Running shorts men lightweight quick dry breathable athletic training",
    "Yoga pants women high waist stretchy comfortable athletic leggings workout",
    "Leather jacket men motorcycle classic black zip closure warm winter",
    "Down winter jacket women warm lightweight packable puffer insulated",
    "Sony WH-1000XM5 wireless noise cancelling headphones over ear bluetooth",
    "Apple AirPods Pro wireless earbuds noise cancelling spatial audio",
]

bm25 = BM25(k1=1.5, b=0.75)
bm25.fit(corpus)

print(f"Corpus: {bm25.corpus_size} documents")
print(f"Average document length: {bm25.avgdl:.1f} tokens")
print(f"Vocabulary size: {len(bm25.idf)} terms")

In [ ]:
# Run some queries
queries = [
    "running shoes men",
    "casual dress women",
    "noise cancelling headphones",
    "Nike shoes",
    "warm winter jacket",
]

for query in queries:
    print(f"\nQuery: '{query}'")
    print("-" * 60)
    results = bm25.search(query, top_k=3)
    for rank, (doc_idx, score) in enumerate(results, 1):
        print(f"  {rank}. [{score:.3f}] {corpus[doc_idx][:70]}...")

---
## Learning to Rank

BM25 gives us a good starting point, but we can do much better by learning a ranking function from user behavior data.

### Three Approaches

#### 1. Pointwise

Treat ranking as a regression/classification problem.

- **Input**: (query, document) feature vector
- **Label**: relevance score (e.g., from human judgment: 0-4, or binary click/no-click)
- **Loss**: MSE (regression) or cross-entropy (classification)
- **Model**: Any regression/classification model (LR, GBDT, DNN)

**Pros**: Simple, standard ML problem  
**Cons**: Ignores relative order of documents. Position bias in labels.

#### 2. Pairwise (RankNet, LambdaRank)

Learn to order pairs of documents correctly.

- **Input**: two (query, document) feature vectors
- **Label**: which document is more relevant
- **Loss**: binary cross-entropy on pair ordering

**RankNet loss**:

Given scores $s_i, s_j$ for documents $d_i, d_j$ where $d_i$ is more relevant:

$$P(d_i \succ d_j) = \sigma(s_i - s_j)$$

$$\mathcal{L} = -\log \sigma(s_i - s_j)$$

**LambdaRank**: Weights the gradient of RankNet by the change in NDCG from swapping $d_i$ and $d_j$. This directly optimizes NDCG (approximately).

$$\lambda_{ij} = \frac{-\sigma(s_j - s_i)}{1 + e^{\sigma(s_i - s_j)}} \cdot |\Delta \text{NDCG}_{ij}|$$

#### 3. Listwise (LambdaMART, SoftmaxLoss)

Optimize the ranking of the entire list directly.

- **Input**: full list of (query, document) feature vectors for a query
- **Loss**: directly related to list-level metric (NDCG, MAP)

**LambdaMART**: GBDT trained with LambdaRank gradients. Industry workhorse for search.

**Comparison**:

| Approach | Pros | Cons | Common Models |
|----------|------|------|---------------|
| Pointwise | Simple | Ignores structure | LR, GBDT |
| Pairwise | Better ranking quality | O(n^2) pairs | RankNet, LambdaRank |
| Listwise | Best ranking quality | Complex, harder to train | LambdaMART, ListNet |

---
## Implementation: Pairwise Ranking Loss (RankNet)

In [ ]:
import numpy as np


class SimpleRankNet:
    """
    Simple pairwise ranking model using a single-layer scoring network.
    
    The model learns a scoring function s(x) = w^T x + b, then trains
    using the RankNet pairwise loss:
        L = -log(sigmoid(s_i - s_j))  where d_i is preferred over d_j
    
    This is a from-scratch NumPy implementation for educational purposes.
    """
    
    def __init__(self, n_features: int, lr: float = 0.01):
        self.n_features = n_features
        self.lr = lr
        # Initialize weights
        np.random.seed(42)
        self.w = np.random.randn(n_features) * 0.01
        self.b = 0.0
    
    def _sigmoid(self, x: np.ndarray) -> np.ndarray:
        """Numerically stable sigmoid."""
        return np.where(
            x >= 0,
            1 / (1 + np.exp(-x)),
            np.exp(x) / (1 + np.exp(x))
        )
    
    def score(self, X: np.ndarray) -> np.ndarray:
        """Compute scores for feature matrix X (n_samples x n_features)."""
        return X @ self.w + self.b
    
    def fit(self, X_pos: np.ndarray, X_neg: np.ndarray, n_epochs: int = 100,
            verbose: bool = True) -> 'SimpleRankNet':
        """
        Train on pairs where X_pos[i] should be ranked above X_neg[i].
        
        X_pos: (n_pairs x n_features) — feature vectors of preferred documents
        X_neg: (n_pairs x n_features) — feature vectors of non-preferred documents
        """
        n_pairs = X_pos.shape[0]
        
        for epoch in range(n_epochs):
            # Forward pass
            s_pos = self.score(X_pos)  # (n_pairs,)
            s_neg = self.score(X_neg)  # (n_pairs,)
            diff = s_pos - s_neg       # should be positive
            
            # RankNet loss: -log(sigmoid(s_pos - s_neg))
            prob = self._sigmoid(diff)
            loss = -np.mean(np.log(prob + 1e-10))
            
            # Accuracy: fraction of pairs correctly ordered
            acc = np.mean(diff > 0)
            
            # Backward pass
            # d(loss)/d(diff) = sigmoid(diff) - 1 = -(1 - sigmoid(diff))
            # But actually: d(-log(sigmoid(x)))/dx = -(1 - sigmoid(x)) = sigmoid(-x) - 1
            # Wait, let's be precise:
            # L = -log(sigmoid(s_i - s_j))
            # dL/d(s_i - s_j) = -(1 - sigmoid(s_i - s_j)) = sigmoid(s_j - s_i) 
            grad_diff = self._sigmoid(-diff)  # (n_pairs,) — this is sigmoid(s_neg - s_pos)
            # But we want gradient to MINIMIZE loss, and the sign above gives us
            # the gradient for (s_pos - s_neg). So:
            # dL/d(w) = grad_diff * (x_pos - x_neg) / n_pairs  (negative because we want descent)
            
            grad_w = -(grad_diff[:, None] * (X_pos - X_neg)).mean(axis=0)
            grad_b = -grad_diff.mean()
            
            # Actually let's re-derive cleanly:
            # L = -log(sigmoid(s_pos - s_neg)) = log(1 + exp(-(s_pos - s_neg)))
            # dL/ds_pos = -(1 - sigma(s_pos-s_neg)) = -sigma(s_neg-s_pos)
            # dL/ds_neg = (1 - sigma(s_pos-s_neg)) = sigma(s_neg-s_pos)
            # s_pos = w^T x_pos + b, so dL/dw = dL/ds_pos * x_pos + dL/ds_neg * x_neg
            lambda_ij = self._sigmoid(s_neg - s_pos)  # (n_pairs,)
            grad_w = (-lambda_ij[:, None] * X_pos + lambda_ij[:, None] * X_neg).mean(axis=0)
            grad_b = 0  # symmetric, cancels out approximately
            
            # Update
            self.w -= self.lr * grad_w
            self.b -= self.lr * grad_b
            
            if verbose and (epoch + 1) % 10 == 0:
                print(f"Epoch {epoch+1:3d} | Loss: {loss:.4f} | Pairwise Accuracy: {acc:.4f}")
        
        return self
    
    def rank(self, X: np.ndarray) -> np.ndarray:
        """Return indices sorted by descending score."""
        scores = self.score(X)
        return np.argsort(scores)[::-1]

In [ ]:
# ============================================================
# Synthetic learning-to-rank experiment
# ============================================================

np.random.seed(42)

# Simulate query-document features for a search ranking task.
# Features: [bm25_score, query_doc_embedding_sim, doc_popularity, doc_freshness, title_match]
n_features = 5
n_queries = 100
docs_per_query = 20

# True relevance weights (unknown to the model)
true_weights = np.array([0.3, 0.4, 0.1, 0.05, 0.15])

# Generate data
all_X_pos = []
all_X_neg = []

for q in range(n_queries):
    # Generate features for documents in this query
    X_query = np.random.rand(docs_per_query, n_features)
    # True relevance = features @ true_weights + noise
    relevance = X_query @ true_weights + np.random.randn(docs_per_query) * 0.05
    
    # Create pairs: sample pairs where we know the ordering
    for _ in range(10):  # 10 pairs per query
        i, j = np.random.choice(docs_per_query, 2, replace=False)
        if relevance[i] > relevance[j]:
            all_X_pos.append(X_query[i])
            all_X_neg.append(X_query[j])
        else:
            all_X_pos.append(X_query[j])
            all_X_neg.append(X_query[i])

X_pos = np.array(all_X_pos)
X_neg = np.array(all_X_neg)
print(f"Training pairs: {len(X_pos)}")
print(f"Feature names: [bm25_score, embedding_sim, popularity, freshness, title_match]")

In [ ]:
# Train the RankNet model
ranker = SimpleRankNet(n_features=n_features, lr=0.5)
ranker.fit(X_pos, X_neg, n_epochs=100)

In [ ]:
# Compare learned weights with true weights
feature_names = ["bm25_score", "embedding_sim", "popularity", "freshness", "title_match"]

print("Feature importance comparison:")
print(f"{'Feature':<16} {'True Weight':>12} {'Learned Weight':>14}")
print("-" * 44)

# Normalize learned weights for comparison
learned_normalized = np.abs(ranker.w) / np.sum(np.abs(ranker.w))
true_normalized = true_weights / np.sum(true_weights)

for name, tw, lw in zip(feature_names, true_normalized, learned_normalized):
    print(f"{name:<16} {tw:>12.3f} {lw:>14.3f}")

In [ ]:
# Evaluate: NDCG on a test query
def ndcg_at_k(scores, relevances, k):
    """Compute NDCG@k given model scores and true relevances."""
    # Rank by model scores
    order = np.argsort(scores)[::-1][:k]
    ranked_rels = relevances[order]
    
    # DCG
    dcg = sum((2**rel - 1) / np.log2(i + 2) for i, rel in enumerate(ranked_rels))
    
    # Ideal DCG
    ideal_rels = np.sort(relevances)[::-1][:k]
    idcg = sum((2**rel - 1) / np.log2(i + 2) for i, rel in enumerate(ideal_rels))
    
    return dcg / idcg if idcg > 0 else 0.0


# Test on fresh queries
ndcgs_model = []
ndcgs_random = []
ndcgs_bm25_only = []

for _ in range(200):
    X_test = np.random.rand(docs_per_query, n_features)
    true_rel = X_test @ true_weights  # true relevance
    
    model_scores = ranker.score(X_test)
    random_scores = np.random.randn(docs_per_query)
    bm25_scores = X_test[:, 0]  # BM25 feature only
    
    ndcgs_model.append(ndcg_at_k(model_scores, true_rel, k=5))
    ndcgs_random.append(ndcg_at_k(random_scores, true_rel, k=5))
    ndcgs_bm25_only.append(ndcg_at_k(bm25_scores, true_rel, k=5))

print(f"Mean NDCG@5:")
print(f"  RankNet model:  {np.mean(ndcgs_model):.4f}")
print(f"  BM25 only:      {np.mean(ndcgs_bm25_only):.4f}")
print(f"  Random:         {np.mean(ndcgs_random):.4f}")

---
## Feature Engineering for Search Ranking

### Query-Document Relevance Features

| Feature | Description |
|---------|------------|
| BM25 score | Lexical relevance |
| Cosine similarity (dense embeddings) | Semantic relevance |
| Exact title match | Binary: does query match product title? |
| Query term coverage | Fraction of query terms found in document |
| TF-IDF cosine | Classical text similarity |

### Document Quality Features

| Feature | Description |
|---------|------------|
| Historical CTR | Click-through rate for this document |
| Sales volume | How popular is this product? |
| Average rating | Product quality signal |
| Number of reviews | Confidence in rating |
| Freshness | How recently was this listed? |

### User Features (Personalization)

| Feature | Description |
|---------|------------|
| User purchase history similarity | Has user bought similar items? |
| User price preference | Does price match user's typical range? |
| User brand affinity | Has user interacted with this brand before? |
| User location | For location-relevant products |

### Context Features

| Feature | Description |
|---------|------------|
| Device type | Mobile vs desktop (affects display) |
| Time of day | Shopping patterns differ |
| Session features | What has user searched/clicked this session? |

---
## Stages 6-8: Training, Serving, Monitoring

### Training Pipeline (Stage 6)

- **Training data**: click logs with position de-biasing (inverse propensity weighting)
- **Label construction**: clicked + purchased = highly relevant, clicked + no purchase = relevant, not clicked = unknown (not necessarily irrelevant!)
- **Negative sampling**: shown-but-not-clicked is a weak negative. Consider impression-based negatives.
- **Retraining**: daily, on a sliding 30-day window
- **Model**: LambdaMART (GBDT) is the industry standard. DNN rankers (BERT-based cross-encoders) for quality but higher latency.

### Position Bias

Users click higher-ranked results more, regardless of relevance. This biases training data.

**Solutions**:
- Inverse Propensity Weighting: weight each click by 1/P(examine | position)
- Position as a feature during training, remove during serving
- Randomized experiments: occasionally shuffle results to collect unbiased data

### Serving Architecture (Stage 7)

```
Query → Query Understanding (5ms)
   → Retrieval: BM25 + ANN (20ms)
     → Feature Computation (30ms)
       → Ranking Model (50ms)
         → Re-ranking + Business Rules (10ms)
           → Response (total ~120ms)
```

### Monitoring (Stage 8)

- **Real-time**: query latency, error rate, zero-result rate
- **Hourly**: CTR by query category, abandonment rate
- **Daily**: NDCG on human-judged queries, feature drift, model freshness
- **Alert on**: >5% drop in CTR, >10% increase in zero-result rate, latency > 300ms at p99

---
## Key Interview Talking Points

1. **Hybrid retrieval is standard**: BM25 (lexical) + ANN (semantic). Explain why both are needed.
2. **Position bias is critical**: always discuss how you handle it in training data.
3. **LambdaMART is the workhorse**: GBDT with LambdaRank gradients. Know why it works.
4. **Offline-online gap**: NDCG improvement offline doesn't always translate to CTR improvement. Discuss interleaving experiments.
5. **Query understanding is underrated**: spell correction and intent classification have outsized impact on user experience.
6. **Latency budget matters**: know how to allocate time across retrieval, ranking, and post-processing.